# L9b Lab: Let's Explore the Perceptron on Non-Linearly Separable Data

In this lab, we'll investigate the performance of the Perceptron on datasets that are not linearly separable, particularly the XOR problem. XOR (exclusive or) is true when exactly one of its two inputs is true. The XOR problem is a classic example that demonstrates the limitations of linear classifiers.

> __Backstory:__ In 1969, AI researchers [Marvin Minsky](https://en.wikipedia.org/wiki/Marvin_Minsky) and [Seymour Papert](https://en.wikipedia.org/wiki/Seymour_Papert) published a book called [Perceptrons](https://en.wikipedia.org/wiki/Perceptrons_(book)), in which they proved that [the Perceptron built by Rosenblatt](https://en.wikipedia.org/wiki/Perceptron) was incapable of learning certain functions, particularly the XOR function. This limitation became a central point in their critique. Soon after the book was published, the first [AI winter](https://en.wikipedia.org/wiki/AI_winter), a period of reduced funding for AI research, began. Whether these events are correlated remains an interesting historical question.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Generate datasets with different separability properties:__ A labeling function assigns a class to each point, and the arrangement of the labels decides whether a straight line can separate the classes. We'll label one cloud of points in three ways and plot each dataset.
> * __Implement and evaluate the Perceptron classifier:__ The Perceptron learns its parameters from training data, and we judge it on test data it has not seen. We'll complete its training loop and score each classifier with a confusion matrix.
> * __Understand the limitations of linear classifiers:__ When the classes are not linearly separable, every pass of the Perceptron through the training data makes at least one mistake. We'll compare the learned boundaries and the test performance on the three datasets.

In [the L9a lecture](../L9a/CHEME-5800-L9a-Lecture-LinearClassificationAndPerceptron-Fall-2026.ipynb), we saw that the Perceptron finds a separating hyperplane when the data are linearly separable. First, we'll build a dataset that _should work_, i.e., it is linearly separable, and then two datasets that we know for sure _will not work_, and see what happens.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

This lab's data generator is in [src/Utility.jl](src/Utility.jl), and its plotting functions are in [src/Visualize.jl](src/Visualize.jl). Your Perceptron goes in [src/Compute.jl](src/Compute.jl). We call [the `L9bXOR.my_perceptron(...)` function](src/Compute.jl) by its qualified name, so that re-running the setup cell picks up your edits.

___

## Task 1: Generate and Visualize the Datasets

In this task, we generate one cloud of points, label it in three ways, and plot the three datasets.

First, let's set some constants we'll need for the data generation and training below. Please look at the comment next to each constant for a description of what it is, permissible values, etc.

In [ ]:
number_of_points = 2000; # number of points in the cloud (a positive integer)
number_of_training_examples = 1200; # points used for training, from 1 to number_of_points - 1; the rest are for testing
c̄ = (0.0, 0.0); # center of the cloud: fixed
ϕ = 60*(π/180); # rotation angle (radians)
maxiter = 1000; # maximum number of passes through the training data, a positive integer (T in L9a)

Each dataset has two continuous features $\mathbf{x}\in\mathbb{R}^{2}$ and a categorical label $y\in\{-1,1\}$. We'll build three label functions $L:\mathbb{R}\times\mathbb{R}\to\{\text{true, false}\}$; a point gets the label $y=1$ when $L(x_{1},x_{2})$ is true, and $y=-1$ otherwise. The XOR pattern uses [the `xor(...)` function](https://docs.julialang.org/en/v1/base/math/#Base.xor):

In [ ]:
# labeling functions: true gives the label 1, false gives the label -1 -
L₁(x₁,x₂) = (x₂ ≥ 0) ? true : false; # half circle
L₂(x₁,x₂) = (x₁ ≥ 0) && (x₂ ≥ 0) ? true : false; # wedge pattern
L₃(x₁,x₂) = xor(x₁ ≥ 0, x₂ ≥ 0); # XOR pattern, alternating pie wedges

# name each pattern, and look up its labeling function by name -
patterns = ["half circle", "wedge", "XOR"]; # the order we'll show the patterns in
labeling_functions = Dict("half circle" => L₁, "wedge" => L₂, "XOR" => L₃); # pattern => labeling function

Now let's generate the datasets. We draw one cloud of `number_of_points` random points in a disk of radius one around `c̄` using [the `generatedatacloud(...)` function](src/Utility.jl), label it with each labeling function, and rotate the points by the angle $\phi$ so the label boundaries do not line up with the axes. Each dataset in the `datasets::Dict{String, Array{Float64,2}}` dictionary holds the two features and the label in its three columns. The cell also shuffles the row numbers into `shuffled_rows::Array{Int64,1}`, which we'll use to split the data:

In [ ]:
datasets, shuffled_rows = let

    # initialize -
    Random.seed!(5800); # fix the random numbers, so we get the same points every time
    cloud = generatedatacloud(c̄, number_of_points = number_of_points, label = 0); # label = 0 is a placeholder; we set the labels below
    R = [
        cos(ϕ) -sin(ϕ) ;
        sin(ϕ) cos(ϕ) ;
    ]; # rotation matrix: turns a point counter-clockwise by ϕ
    datasets = Dict{String, Array{Float64,2}}();

    # main loop: label the same cloud with each labeling function -
    for pattern ∈ patterns
        L = labeling_functions[pattern]; # which labeling function?
        D = copy(cloud); # copy, so each dataset gets its own labels

        # compute the label by taking the XOR, AND, etc functions -
        for i ∈ 1:number_of_points
            flag = L(cloud[i,1], cloud[i,2]);
            if (flag == true)
                D[i,3] = 1
            else
                D[i,3] = -1
            end
        end

        # keep the label, but apply the rotation to the (x₁,x₂) data -
        for i ∈ 1:number_of_points
            D[i,1:2] = R*cloud[i,1:2];
        end

        datasets[pattern] = D;
    end

    # shuffle the row numbers here, so re-running the split gives the same training and test points -
    shuffled_rows = randperm(number_of_points); # the row numbers 1, 2, ..., n in a random order

    datasets, shuffled_rows # return
end;

Let's plot the three datasets side by side. [The `plot_dataset(...)` function](src/Visualize.jl) draws each point at its two features, in arbitrary units (AU), colored by its label: orange for label `1` and teal for label `-1`.

__Prediction:__ Which of the three datasets can a single straight line separate?

In [ ]:
let

    # initialize -
    panels = []; # one figure per pattern

    # make a scatter plot for each pattern -
    for pattern ∈ patterns
        push!(panels, plot_dataset(datasets[pattern], title = pattern));
    end

    # draw the three panels side by side -
    plot(panels..., layout = (1,3), size = (1200, 440)) # ... passes each panel to plot as its own argument
end

__What do we see?__ The three datasets hold the same points; only the labels differ. A straight line through the center separates the half-circle labels, but no straight line separates the wedge or the XOR labels.

Next, let's split each dataset into `training` and `test` subsets:

* __Training data__: Training datasets are collections of labeled data used to teach machine learning models. In our case, we'll use the training data to estimate the classifier parameters $\mathbf{\theta}$.
* __Test data__: Test datasets are separate labeled data, used to estimate how well a trained model works on new data drawn in the same way.

We do this randomly: the first `number_of_training_examples::Int64` rows of `shuffled_rows` are for training, and the rest are for testing; all three datasets use the same training points. The shuffle happened in the seeded data cell, so re-running the split gives the same training and test points. We store the subsets in the `training::Dict{String, Array{Float64,2}}` and `test::Dict{String, Array{Float64,2}}` dictionaries:

In [ ]:
training, test = let

    # initialize -
    training_rows = shuffled_rows[1:number_of_training_examples]; # the first shuffled rows, for training
    test_rows = shuffled_rows[(number_of_training_examples + 1):end]; # the remaining rows, for testing
    training = Dict{String, Array{Float64,2}}();
    test = Dict{String, Array{Float64,2}}();

    # select the same rows from each dataset -
    for pattern ∈ patterns
        D = datasets[pattern]; # which dataset are we splitting?
        training[pattern] = D[training_rows, :];
        test[pattern] = D[test_rows, :];
    end

    training, test # return
end;

### Things to think about

__Question:__ For the wedge pattern, where would you draw the best straight line, and which points would it get wrong? Does rotating the points change whether a dataset is linearly separable? Briefly explain.

___

## Task 2: Build a Perceptron Classification Model and Learn the Parameters

In this task, we'll build a model of our classification problem and train the model using an online learning method, one model for each training set. Online means the method visits the training examples one at a time and updates the parameters right after each mistake.

* __Training__: Our Perceptron implementation, which follows the L9a pseudocode, stores problem information in [a `MyPerceptronClassificationModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyPerceptronClassificationModel), which holds the parameters, starting from zero, and the mistake threshold $M$.
* Next, we _learn_ the model parameters [using the `learn(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.learn), which takes the training features `X`, the training labels `y`, and the model, and returns the model with updated parameters. The `maxiter` argument is $T$ in the lecture, the maximum number of passes through the training data.

We store the trained models in the `perceptron_models::Dict{String, MyPerceptronClassificationModel}` dictionary; each model holds $\mathbf{\theta}$ in its `β` field.

__Prediction:__ For which datasets will the Perceptron stop with zero mistakes on the training data, and for which will it use all of its passes?

In [ ]:
perceptron_models = let

    # initialize -
    models = Dict{String, MyPerceptronClassificationModel}();

    # main loop: train one Perceptron for each pattern -
    for pattern ∈ patterns

        # setup -
        D = training[pattern]; # what dataset are we going to use?
        number_of_examples = size(D,1); # how many examples do we have (rows)
        X = [D[:,1:end-1] ones(number_of_examples)]; # features, what??
        y = D[:,end]; # output: this is the target data (label)
        number_of_parameters = size(X,2); # two weights and the bias

        # build an initial model -
        model = build(MyPerceptronClassificationModel, (
            parameters = zeros(number_of_parameters), # start from θ = 0
            mistakes = 0 # willing to live with m mistakes
        ));

        # train the model -
        print("$(pattern): "); # label the message that learn(...) prints
        models[pattern] = learn(X, y, model, maxiter = maxiter, verbose = true);
    end

    models # return
end;

__What do we see?__ Each iteration in the printed message is one pass through the training data, and the errors are the mistakes in the last pass. With the constants above, the half-circle loop stopped after five passes, the fifth with no mistakes. The wedge and XOR loops used all 1000 passes and still made 234 and 610 mistakes in the last pass.

__Inference__: Now that we have parameters estimated from the `training` data, we can use those parameters on the `test` dataset to see how well the model can differentiate between classes on data it has never seen.

[The `classify(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.classify) takes a feature array `X` and the (trained) model instance, and returns the sign of each score. A score of exactly zero gives `0`, so we give those points the label `1`, as in L9a. We store the actual labels in `y_perceptron` and the predicted labels in `ŷ_perceptron`, both of type `Dict{String, Array{Float64,1}}`.

In [ ]:
ŷ_perceptron, y_perceptron = let

    # initialize -
    ŷ_perceptron = Dict{String, Array{Float64,1}}(); # predicted labels for each pattern
    y_perceptron = Dict{String, Array{Float64,1}}(); # actual labels for each pattern

    # classify the test points for each pattern -
    for pattern ∈ patterns
        D = test[pattern]; # what dataset are we going to use?
        number_of_examples = size(D,1); # how many examples do we have (rows)
        X = [D[:,1:end-1] ones(number_of_examples)]; # features: need to add a 1 to each row (for bias), after removing the label
        y = D[:,end]; # output: this is the *actual* target data (label)

        # compute the estimated labels -
        ŷ = classify(X, perceptron_models[pattern]); # the sign of each score
        ŷ[ŷ .== 0] .= 1; # a score of exactly zero gets the label 1, as in L9a
        ŷ_perceptron[pattern] = ŷ;
        y_perceptron[pattern] = y;
    end

    ŷ_perceptron, y_perceptron # return
end;

### Implement the Perceptron

Now let's write the training loop ourselves, following the L9a pseudocode. Open [`src/Compute.jl`](src/Compute.jl) and complete [the `my_perceptron(...)` function](src/Compute.jl). Everything except three lines is already written, and a TODO comment marks each of them:

> __Implementation tasks__
>
> * __TODO 1:__ The mistake test. Replace `false` with $y_{i}\,\hat{\mathbf{x}}_{i}^{\top}\mathbf{\theta}\leq{0}$; [the `dot(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.dot) computes $\hat{\mathbf{x}}_{i}^{\top}\mathbf{\theta}$.
> * __TODO 2:__ The update. Replace the right-hand side of `θ = θ` with $\mathbf{\theta}+y_{i}\,\hat{\mathbf{x}}_{i}$.
> * __TODO 3:__ The stopping test. Replace `false` with step 4 of the pseudocode: the pass made at most $M$ mistakes, or $t$ has reached $T$. The function names the `mistakes` and `maxiter` arguments `M` and `T`, as in L9a.

Save the file and re-run the setup cell at the top of the notebook to reload your edits, then run the cell below. Until your loop returns a result, the call throws an error:

In [ ]:
my_results = let

    # initialize -
    results = Dict{String, NamedTuple}(); # what my_perceptron(...) returns, for each pattern

    # train your Perceptron on each training set; the untouched stub throws an error -
    for pattern ∈ patterns
        D = training[pattern]; # what dataset are we going to use?
        X = [D[:,1:end-1] ones(size(D,1))]; # features, with a 1 for the bias
        y = D[:,end]; # output: this is the target data (label)
        results[pattern] = L9bXOR.my_perceptron(X, y, maxiter = maxiter, mistakes = 0);
    end

    results # return
end;

### Check: Your Perceptron

Your function starts from the same parameters and visits the examples in the same order as the course package, so it should learn the same parameters and stop after the same number of passes.

The last two tests check the mistake threshold on the wedge data. First, your function makes a single pass and counts its mistakes, $m_{1}$. With a threshold of $m_{1}$ or $m_{1}+1$, the first pass has at most that many mistakes, so your loop should stop after it:

In [ ]:
@testset "My Perceptron" begin

    # the same parameters as the package, for each pattern -
    for pattern ∈ patterns
        @test my_results[pattern].θ ≈ perceptron_models[pattern].β
    end

    # the half circle stops early, with no mistakes; the wedge and XOR loops use every pass -
    @test my_results["half circle"].mistakes == 0 && my_results["half circle"].passes < maxiter
    @test my_results["wedge"].passes == maxiter && my_results["wedge"].mistakes > 0
    @test my_results["XOR"].passes == maxiter && my_results["XOR"].mistakes > 0

    # mistake thresholds, on the wedge data: one pass first, to count its mistakes, m₁ -
    D = training["wedge"]; # the wedge training data
    X = [D[:,1:end-1] ones(size(D,1))]; # features, with a 1 for the bias
    y = D[:,end]; # output: this is the target data (label)
    m₁ = L9bXOR.my_perceptron(X, y, maxiter = 1).mistakes; # the mistakes in the first pass

    # with a threshold of m₁ or m₁ + 1, the first pass has at most that many mistakes, so stop -
    for M ∈ (m₁, m₁ + 1)
        result = L9bXOR.my_perceptron(X, y, maxiter = maxiter, mistakes = M);
        @test (result.passes, result.mistakes) == (1, m₁)
    end
end;

### Things to think about

__Question:__ Why does the Perceptron stop after a few passes on the half-circle data but use all 1000 passes on the wedge and XOR data? Would 10,000 passes let it reach zero mistakes on the wedge data? Briefly explain.

___

## Task 3: Evaluate the Classifiers and Visualize the Misses

In this task, we compute the confusion matrix and the L9a performance metrics for each classifier, and plot which test points each classifier gets wrong.

__Confusion matrix__: The confusion matrix is a $2\times{2}$ matrix that contains four entries, laid out as in the L9a lecture: true positive (TP), false negative (FN), false positive (FP), and true negative (TN). Label `1` is the positive class.

Let's compute the confusion matrix [using the `confusion(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.confusion) and store the results in the `CM_perceptron::Dict{String, Array{Int64,2}}` dictionary:

In [ ]:
CM_perceptron = let

    # initialize -
    CM_perceptron = Dict{String, Array{Int64,2}}();

    # compute the confusion matrix for each pattern -
    for pattern ∈ patterns
        CM_perceptron[pattern] = confusion(y_perceptron[pattern], ŷ_perceptron[pattern]); # [TP FN; FP TN]
    end

    CM_perceptron # return
end;

Finally, we can compute the accuracy, precision, recall, and specificity from the L9a lecture using the entries of each confusion matrix.

__Prediction:__ What test accuracy do you expect for each dataset? For the XOR data, will the Perceptron do {better than, about the same as, or worse than} guessing at random?

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per pattern

    # compute the performance metrics from each confusion matrix -
    for pattern ∈ patterns
        CM = CM_perceptron[pattern]; # which confusion matrix?
        TP = CM[1,1]; # true positives
        FN = CM[1,2]; # false negatives
        FP = CM[2,1]; # false positives
        TN = CM[2,2]; # true negatives

        # add a row to the table -
        push!(df, (pattern = pattern, TP = TP, FN = FN, FP = FP, TN = TN,
            accuracy = round((TP + TN)/(TP + TN + FP + FN), digits = 3),
            precision = round(TP/(TP + FP), digits = 3),
            recall = round(TP/(TP + FN), digits = 3),
            specificity = round(TN/(TN + FP), digits = 3)));
    end

    # let's write a pretty table of the metrics for each pattern -
    pretty_table(df; backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact),
        column_labels = ["Pattern", "TP", "FN", "FP", "TN", "Accuracy", "Precision", "Recall", "Specificity"],
        alignment = :r,
        fit_table_in_display_horizontally = false, # show every column
    );
end

__What do we see?__ With our seed and split, the half-circle classifier missed one of the 800 test points, and the wedge classifier scored an accuracy of 0.831. On the XOR data, the accuracy is 0.482, about what we would get by guessing at random.

__Visualize the misses__: Using the test dataset, let's show (with gray circles) which samples our classifier is unable to predict the label correctly, i.e., where we miss the label. [The `plot_misses(...)` function](src/Visualize.jl) also draws the learned decision boundary, the line where $w_{1}x_{1}+w_{2}x_{2}+b=0$:

In [ ]:
let

    # initialize -
    panels = []; # one figure per pattern

    # draw the misses and the learned boundary for each pattern -
    for pattern ∈ patterns
        θ = perceptron_models[pattern].β; # learned parameters: w₁, w₂, then the bias b
        push!(panels, plot_misses(test[pattern], ŷ_perceptron[pattern], θ, title = "Perceptron: $(pattern)"));
    end

    # draw the three panels side by side -
    plot(panels..., layout = (1,3), size = (1200, 470)) # ... passes each panel to plot as its own argument
end

__What do we see?__ For the half circle, the learned line runs along the boundary between the labels, and the one gray point lies right next to it. For the wedge, the line cuts across the orange quarter. For the XOR data, each side of the line holds points of both labels, so roughly half of the points on each side are gray.

### Things to think about

__Question:__ Which kind of mistake does the wedge classifier make most often, a false positive or a false negative? Why is its accuracy misleading? For the XOR data, can you think of a new feature, computed from the two coordinates before the rotation, that would make the labels linearly separable?

___

## Summary

In this lab, we explored the Perceptron algorithm and its fundamental limitations on non-linearly separable datasets.

> __Key Takeaways:__
>
> * __Separability depends on the labels:__ We labeled the same cloud of points in three ways and plotted the results. Only the half-circle labels can be separated by a straight line.
>
> * __Linear classifiers work on linearly separable data:__ Our Perceptron matched the course package and stopped after a few passes on the half-circle data, with no training mistakes. It then classified all but one of the test points correctly.
>
> * __The XOR problem exposes the limits of a linear classifier:__ On the wedge and XOR data, the Perceptron used every pass without reaching zero mistakes, and on XOR its test accuracy was close to random guessing. On the wedge, a high accuracy hid a classifier that missed most of the points with label 1.

__Where do we go from here?__ The limitations we observed with the Perceptron on non-linearly separable problems motivated the development of multi-layer neural networks, which stack layers of Perceptron-like units, and other non-linear classifiers. These methods can learn curved decision boundaries by transforming the input features, for example by multiplying two features together. We'll take a look at these in CHEME 5820 next semester.

___